In [2]:
from two_tower_confounding.models.towers import *
from two_tower_confounding.metrics import NDCG, MRR, NegativeLogLikelihood
from two_tower_confounding.models.two_tower import TwoTowerModel
from two_tower_confounding.simulation.simulator import Simulator
from two_tower_confounding.trainer import Trainer
from two_tower_confounding.utils import np_collate

/opt/homebrew/Caskroom/miniforge/base/envs/two-tower-confounding-2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# load: two-tower-confounding/results/test/test_datasets/test_click_dataset_policy_temperature0.0_sdoc0.0_num_queries1_.pkl

import pickle as pkl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import two_tower_confounding as ttc

with open('results/test/test_datasets/test_dataset_policy_temperature0.0_sdoc0.0_num_queries1_.pkl', 'rb') as f:
    click_data = pkl.load(f)
with open('results/test/test_datasets/test_clidataset_policy_temperature1.0_sdoc0.0_num_queries1_.pkl', 'rb') as f:
    click_data_tmp_1 = pkl.load(f)


In [14]:
df = pd.DataFrame(list(click_data))
df_tmp_1 = pd.DataFrame(list(click_data_tmp_1))

In [20]:
df_tmp_1

,query,query_doc_ids,query_doc_features,lp_query_doc_features,labels,mask,n
0,0,"(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[True, True, True, True, True, True, True, Tru...",10


In [15]:
df.query_doc_ids = df.query_doc_ids.apply(lambda x: tuple(x))
df.query_doc_ids.unique()

array([(np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10))],
      dtype=object)

In [16]:
df

,query,query_doc_ids,query_doc_features,lp_query_doc_features,labels,mask,n
0,0,"(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)","[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","[[0.0, 1.741676], [0.0, 1.5654233], [0.0, 1.52...","[0.0, 0.2965051492107924, 0.4196948181788126, ...","[True, True, True, True, True, True, True, Tru...",10


In [18]:
df_tmp_1.query_doc_ids = df.query_doc_ids.apply(lambda x: tuple(x))
df_tmp_1.query_doc_ids.unique()

array([(np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10))],
      dtype=object)

In [19]:
df_tmp_1.query_doc_ids

0    (1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
Name: query_doc_ids, dtype: object

In [68]:
df

,query,query_doc_features,query_doc_ids,labels,propensities,clicks,positions,mask,n
0,4,"[[0.7015617, 0.5488148, 0.60201824, 0.24623486...","(41, 42, 43, 44, 45, 46, 47, 48, 49, 50)","[1.3039911655389678, 1.3039911655389678, 1.303...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
1,8,"[[-0.89641446, 0.5857847, -0.2175278, 0.133393...","(81, 82, 83, 84, 85, 86, 87, 88, 89, 90)","[1.8963089162004436, 1.8963089162004436, 1.896...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
2,1,"[[-0.3590721, 0.67289346, -0.15888232, -0.1986...","(11, 12, 13, 14, 15, 16, 17, 18, 19, 20)","[2.585823372687536, 2.585823372687536, 2.58582...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 0, 0, 1, 1, 0, 0, 0, 1, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
3,1,"[[-0.3590721, 0.67289346, -0.15888232, -0.1986...","(11, 12, 13, 14, 15, 16, 17, 18, 19, 20)","[2.585823372687536, 2.585823372687536, 2.58582...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 1, 0, 1, 0, 0, 0, 1, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
4,8,"[[-0.89641446, 0.5857847, -0.2175278, 0.133393...","(81, 82, 83, 84, 85, 86, 87, 88, 89, 90)","[1.8963089162004436, 1.8963089162004436, 1.896...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
...,...,...,...,...,...,...,...,...,...
49995,1,"[[-0.3590721, 0.67289346, -0.15888232, -0.1986...","(11, 12, 13, 14, 15, 16, 17, 18, 19, 20)","[2.585823372687536, 2.585823372687536, 2.58582...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 1, 0, 0, 0, 1, 0, 1, 1, 1]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49996,2,"[[0.21244654, -0.43061292, 0.25278002, 0.48217...","(21, 22, 23, 24, 25, 26, 27, 28, 29, 30)","[2.8512289886567426, 2.8512289886567426, 2.851...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 0, 0, 0, 0, 0, 1, 0, 1, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49997,7,"[[0.018036364, -0.33590063, -0.24537462, -0.38...","(71, 72, 73, 74, 75, 76, 77, 78, 79, 80)","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
49998,8,"[[-0.89641446, 0.5857847, -0.2175278, 0.133393...","(81, 82, 83, 84, 85, 86, 87, 88, 89, 90)","[1.8963089162004436, 1.8963089162004436, 1.896...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]","[True, True, True, True, True, True, True, Tru...",10
